
# Plotting combined score of samples in feature space

Visualization of the representations learned via :class:`pan.pan.ParallelAnomalousNudge` and the obtained combined score of samples.


In [ ]:
import numpy as np
import matplotlib as mpl
from matplotlib import pyplot as plt
from pan import ParallelAnomalousNudge

np.set_printoptions(suppress=True)
RANDOM_SEED = 42

N_NORMAL = 100
N_ABNORMAL = 20

In [ ]:
# Training data

np.random.seed(42)

# Generate normal samples
f1 = np.random.normal(loc=.06, scale=.002, size=N_NORMAL)
f2 = np.random.normal(loc=.075, scale=.002, size=N_NORMAL)
X_normal = np.vstack((f1, f2)).T

# Generate abnormal samples
f1_abn = np.random.normal(loc=.06, scale=.002, size=N_ABNORMAL) + np.random.uniform(0.005, 0.01, size=N_ABNORMAL)
f2_abn = np.random.normal(loc=.075, scale=.002, size=N_ABNORMAL) + np.random.uniform(0.005, 0.01, size=N_ABNORMAL)
X_abnormal = np.vstack((f1_abn, f2_abn)).T

X_train = np.vstack((X_normal, X_abnormal))
y_train = np.hstack((np.zeros(N_NORMAL), np.ones(N_ABNORMAL)))

In [ ]:
# Train model

estimator = ParallelAnomalousNudge(random_seed=RANDOM_SEED)
estimator.fit(X_train, y_train)

train_scores = estimator.combined_score_samples(X_train)

In [ ]:
from pan.utils.som_display import SomLatticeDisplay

def unscale_som_lattices(estimator:ParallelAnomalousNudge):
    lattices = []
    for j, (est, sclr) in enumerate(zip(estimator.estimators_, map(lambda c: estimator.scalers_[c], estimator.classes_))):
        node_weights = est.som_.get_weights().copy()
        shape = node_weights.shape
        lattices.append(sclr.inverse_transform(node_weights.reshape(-1, shape[-1])).reshape(shape))
    return lattices

# Plot combined score space

fig, ax = plt.subplots(figsize=(11, 6))

ax.scatter(*X_train[y_train == 0].T, **{"marker": ".", "color": "none", "edgecolor": "k", "alpha": 0.5}, label="Normal train", zorder=6)
ax.scatter(*X_train[y_train == 1].T, **{"marker": "^", "color": "none", "edgecolor": "k", "alpha": 0.5}, label="Anomaly train", zorder=6)

unscaled_som_lattices = unscale_som_lattices(estimator)
SomLatticeDisplay().from_values(unscaled_som_lattices[0]) \
    .plot(ax=ax, marker="s", color="k", alpha=.4, linewidth=1.0)
SomLatticeDisplay().from_values(unscaled_som_lattices[1]) \
    .plot(ax=ax, marker="s", color="k", alpha=.4, linewidth=1.0)


cmap=plt.cm.Spectral
norm = mpl.colors.SymLogNorm(linthresh=abs(max(train_scores)), vmin=min(train_scores), vmax=0)
ax.scatter(*X_train.T, marker="s", s=80, c=train_scores, cmap=cmap, norm=norm, alpha=.75, zorder=5)

cax = ax.inset_axes([0.44, 0.06, 0.54, 0.03])
plt.colorbar(mpl.cm.ScalarMappable(cmap=cmap, norm=norm), cax=cax, orientation="horizontal", alpha=.75)


ax.set_aspect("equal")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Test data

np.random.seed(84)

# Generate normal samples
f1 = np.random.normal(loc=.06, scale=.002, size=N_NORMAL)
f2 = np.random.normal(loc=.075, scale=.002, size=N_NORMAL)
X_normal = np.vstack((f1, f2)).T

# Generate abnormal samples
f1_abn = np.random.uniform(.06-0.015, .06+0.015, size=N_ABNORMAL)
f2_abn = np.random.uniform(.075-0.015, .075+0.015, size=N_ABNORMAL)
X_abnormal = np.vstack((f1_abn, f2_abn)).T

X_test = np.vstack((X_normal, X_abnormal))
y_test = np.hstack((np.zeros(N_NORMAL), np.ones(N_ABNORMAL)))

fig, ax = plt.subplots(figsize=(11, 6))

ax.scatter(*X_train[y_train == 0].T, **{"marker": ".", "color": "none", "edgecolor": "k", "alpha": 0.5}, label="Normal train", zorder=6)
ax.scatter(*X_train[y_train == 1].T, **{"marker": "^", "color": "none", "edgecolor": "k", "alpha": 0.5}, label="Anomaly train", zorder=6)

unscaled_som_lattices = unscale_som_lattices(estimator)
SomLatticeDisplay().from_values(unscaled_som_lattices[0]) \
    .plot(ax=ax, marker="s", color="k", alpha=.4, linewidth=1.0)
SomLatticeDisplay().from_values(unscaled_som_lattices[1]) \
    .plot(ax=ax, marker="s", color="k", alpha=.4, linewidth=1.0)

test_scores = estimator.combined_score_samples(X_test)
cmap=plt.cm.Spectral
norm = mpl.colors.SymLogNorm(linthresh=abs(max(test_scores)), vmin=min(test_scores), vmax=0)
ax.scatter(*X_test.T, marker="s", s=80, c=test_scores, cmap=cmap, norm=norm, alpha=.75, zorder=5)

ax.set_aspect("equal")
plt.legend()
plt.tight_layout()
plt.show()